In [18]:
import os
import csv
import re


In [2]:
os.getcwd()

'/home/brantran/power_modeling/accel-sim-framework'

In [21]:
# 1) read the csv of coefficients
coeff_path = os.path.join(os.getcwd(),'util','accelwattch','scaled_coefficients.csv')
# print(coeff_path)
assert(os.path.exists(coeff_path))
scaled_components = [
    "IBP", "ICP", "DCP", "CCP", "SHRDP", "RFP", "INTP", "FPUP", "DPUP",
    "INT_MULP", "FP_MULP", "FP_SQRTP", "FP_LGP", "FP_SINP", "FP_EXP",
    "DP_MULP", "TENSORP", "TEXP", "SCHEDP", "L2CP", "DRAMP", "PIPEP",
    "IDLE_COREP", "CONSTP", "STATICP"
]

with open(coeff_path, 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

assert len(lines) == len(scaled_components), "Mismatch between coefficients and component labels"

coeff_dict = dict(zip(scaled_components, map(float, lines)))
coeff_dict


{'IBP': 0.1,
 'ICP': 500.05,
 'DCP': 500.05,
 'CCP': 15.799,
 'SHRDP': 0.1,
 'RFP': 0.45305,
 'INTP': 0.1,
 'FPUP': 0.1,
 'DPUP': 0.1,
 'INT_MULP': 0.50973,
 'FP_MULP': 0.1,
 'FP_SQRTP': 0.1,
 'FP_LGP': 1.0353,
 'FP_SINP': 0.1,
 'FP_EXP': 0.71075,
 'DP_MULP': 0.1,
 'TENSORP': 0.72602,
 'TEXP': 41.813,
 'SCHEDP': 5.425,
 'L2CP': 500.05,
 'DRAMP': 7.8515,
 'PIPEP': 0.1,
 'IDLE_COREP': 1.0,
 'CONSTP': 1.0,
 'STATICP': 1.0}

In [20]:
# 2) read in xml file

xml_path = os.path.join(os.getcwd(), 'gpu-simulator', 'gpgpu-sim', 'configs',
                        'tested-cfgs', 'SM7_TITANV', 'accelwattch_sass_sim.xml')

param_dict = {}

with open(xml_path, 'r') as f:
    lines = f.readlines()[34:68]  # Python is 0-indexed; lines 35-68 = indexes 34 to 67

param_pattern = re.compile(r'<param name="([^"]+)" value="([^"]+)"')

for line in lines:
    match = param_pattern.search(line)
    if match:
        name, value = match.groups()
        param_dict[name] = float(value)
param_dict
# Example usage:
# print(param_dict["PIPE_A"])

{'TOT_INST': 10.0,
 'FP_INT': 4.661,
 'IC_H': 8.593489331,
 'IC_M': 29.735231,
 'DC_RH': 9.835033124,
 'DC_RM': 10.95446778,
 'DC_WH': 0.679656761,
 'DC_WM': 17.67551799,
 'CC_H': 0.1107,
 'CC_M': 0.1233,
 'SHRD_ACC': 0.779992642,
 'REG_RD': 0.100560581,
 'REG_WR': 0.140604679,
 'INT_ACC': 14.98768151,
 'FP_ACC': 0.529670751,
 'DP_ACC': 0.777229051,
 'INT_MUL_ACC': 0.115098047,
 'FP_MUL_ACC': 0.089517055,
 'FP_SQRT_ACC': 0.195089274,
 'FP_LG_ACC': 0.125521663,
 'FP_SIN_ACC': 0.13336307,
 'FP_EXP_ACC': 0.36204415,
 'DP_MUL_ACC': 0.1321288,
 'TENSOR_ACC': 0.815454621,
 'TEX_ACC': 0.115100088,
 'MEM_RD': 0.025941068,
 'MEM_WR': 0.031443719,
 'MEM_PRE': 0.008647023,
 'L2_RH': 1.260867526,
 'L2_RM': 2.394535301,
 'L2_WH': 4.124916,
 'L2_WM': 1.222707601,
 'NOC_A': 32.09037703,
 'PIPE_A': 0.514}

In [23]:
param_key_map = {
    'TOT_INST': 'IBP',
    'FP_INT': 'SCHEDP',
    'IC_H': 'ICP',
    'IC_M': 'ICP',
    'DC_RH': 'DCP',
    'DC_RM': 'DCP',
    'DC_WH': 'DCP',
    'DC_WM': 'DCP',
    'CC_H': 'CCP',
    'CC_M': 'CCP',
    'SHRD_ACC': 'SHRDP',
    'REG_RD': 'RFP',
    'REG_WR': 'RFP',
    'INT_ACC': 'INTP',
    'FP_ACC': 'FPUP',
    'DP_ACC': 'DPUP',
    'INT_MUL_ACC': 'INT_MULP',
    'FP_MUL_ACC': 'FP_MULP',
    'FP_SQRT_ACC': 'FP_SQRTP',
    'FP_LG_ACC': 'FP_LGP',
    'FP_SIN_ACC': 'FP_SINP',
    'FP_EXP_ACC': 'FP_EXP',
    'DP_MUL_ACC': 'DP_MULP',
    'TENSOR_ACC': 'TENSORP',
    'TEX_ACC': 'TEXP',
    'MEM_RD': 'DRAMP',
    'MEM_WR': 'DRAMP',
    'MEM_PRE': 'DRAMP',
    'L2_RH': 'L2CP',
    'L2_RM': 'L2CP',
    'L2_WH': 'L2CP',
    'L2_WM': 'L2CP',
    'NOC_A': 'L2CP',
    'PIPE_A': 'PIPEP',
}
updated_params_dict = {}
for component,init_value in param_dict.items():
    updated_params_dict[component] = init_value * coeff_dict[param_key_map[component]]
updated_params_dict

{'TOT_INST': 1.0,
 'FP_INT': 25.285924999999995,
 'IC_H': 4297.17433996655,
 'IC_M': 14869.10226155,
 'DC_RH': 4918.008313656201,
 'DC_RM': 5477.781613389,
 'DC_WH': 339.86236333805005,
 'DC_WM': 8838.6427708995,
 'CC_H': 1.7489493,
 'CC_M': 1.9480167,
 'SHRD_ACC': 0.0779992642,
 'REG_RD': 0.04555897122205,
 'REG_WR': 0.06370094982095001,
 'INT_ACC': 1.4987681510000002,
 'FP_ACC': 0.05296707510000001,
 'DP_ACC': 0.0777229051,
 'INT_MUL_ACC': 0.05866892749731,
 'FP_MUL_ACC': 0.0089517055,
 'FP_SQRT_ACC': 0.0195089274,
 'FP_LG_ACC': 0.1299525777039,
 'FP_SIN_ACC': 0.013336307,
 'FP_EXP_ACC': 0.2573228796125,
 'DP_MUL_ACC': 0.01321288,
 'TENSOR_ACC': 0.59203636393842,
 'TEX_ACC': 4.812679979544001,
 'MEM_RD': 0.203676295402,
 'MEM_WR': 0.2468803597285,
 'MEM_PRE': 0.0678921010845,
 'L2_RH': 630.4968063763,
 'L2_RM': 1197.38737726505,
 'L2_WH': 2062.6642458,
 'L2_WM': 611.41493588005,
 'NOC_A': 16046.7930338515,
 'PIPE_A': 0.0514}